## Extend CDR Sequence Files with Calculated Features
---
#### vorher laden:
- 
---

#### woher stammen die Werte:

Hydrophobicity: 
- Kyte J, Doolittle RF. A simple method for displaying the hydropathic character of a protein. J Mol Biol. 1982 May 5;157(1):105-32
---

charge:
- Die Werte basieren auf dem Ladungszustand der Seitenkette bei neutralem pH (~7.0)
- Zuordnung ist biochemisch fest definiert:
=> +1: positiv geladen (Arg, Lys, His)
=> -1: negativ geladen (Asp, Glu)
=> 0: ungeladen (alle anderen)
---

mass: [g/mol]
- https://www.sigmaaldrich.com/DE/de/technical-documents/technical-article/protein-biology/protein-structural-analysis/amino-acid-reference-chart?srsltid=AfmBOoqS9jHqRJYPEYQ19QbSpXnpfRSnIhBC5KOjPBVqHMXzcIJfQvVl
- mittleren molaren Masse der freien Aminosäure aus Tabelle entnommen (Rückstandsgewicht - H_20)
---

polarity:
- Grantham, R. (1974). Amino acid difference formula to help explain protein evolution.
Science, 185(4154), 862–864.
---




In [1]:

# Create properties of the AS in Dictionary

import pandas as pd

amino_acid_properties = {
    'A': {'hydrophobicity': 1.8, 'charge': 0, 'mass': 71.08, 'polarity': 8.1 }, #Alanin
    'R': {'hydrophobicity': -4.5, 'charge': +1, 'mass': 156.19, 'polarity': 10.5},# Arginin
    'N': {'hydrophobicity': -3.5, 'charge': 0, 'mass': 114.11, 'polarity': 11.6},#Asparagin
    'D': {'hydrophobicity': -3.5, 'charge': -1, 'mass': 115.09, 'polarity': 13.0},#Asparaginsäure
    'C': {'hydrophobicity': 2.5, 'charge': 0, 'mass': 103.15, 'polarity': 5.5},#Cystein
    'Q': {'hydrophobicity': -3.5, 'charge': 0, 'mass': 128.13, 'polarity': 10.5},#Glutamin
    'E': {'hydrophobicity': -3.5, 'charge': -1, 'mass': 129.12, 'polarity': 12.3},#Glutaminsäure
    'G': {'hydrophobicity': -0.4, 'charge': 0, 'mass': 75.05, 'polarity': 9.0},#Glycin
    'H': {'hydrophobicity': -3.2, 'charge': +1, 'mass': 137.14, 'polarity': 10.4},#Histidin
    'I': {'hydrophobicity': 4.5, 'charge': 0, 'mass': 113.16, 'polarity': 5.2},#Isoleucin
    'L': {'hydrophobicity': 3.8, 'charge': 0, 'mass': 113.16, 'polarity': 4.9},#Leucin
    'K': {'hydrophobicity': -3.9, 'charge': +1, 'mass': 128.18, 'polarity': 11.3},#Lysin
    'M': {'hydrophobicity': 1.9, 'charge': 0, 'mass': 131.20, 'polarity': 5.7},#Methionin
    'F': {'hydrophobicity': 2.8, 'charge': 0, 'mass': 147.18, 'polarity': 5.2},#Phenylalanin
    'P': {'hydrophobicity': -1.6, 'charge': 0, 'mass': 97.12, 'polarity': 8.0},#Prolin
    'S': {'hydrophobicity': -0.8, 'charge': 0, 'mass': 87.08, 'polarity': 9.2},#Serin
    'T': {'hydrophobicity': -0.7, 'charge': 0, 'mass': 101.11, 'polarity': 8.6},#Threonim
    'W': {'hydrophobicity': -0.9, 'charge': 0, 'mass': 186.22, 'polarity': 5.4},#Tryptophan
    'Y': {'hydrophobicity': -1.3, 'charge': 0, 'mass': 163.18, 'polarity': 6.2},#Tyrosin
    'V': {'hydrophobicity': 4.2, 'charge': 0, 'mass': 99.13, 'polarity': 5.9}#Valin
}

### Clean up CDR sequences and calculate the number of amino acids

In [2]:


# Amino acid properties 
amino_acids = list(amino_acid_properties.keys())

# Function for checking whether the sequence is valid
def is_valid_sequence(seq):
    return isinstance(seq, str) and seq.strip().lower() not in ["", "nan", "none"]



# Paths to the file - with folder names
files = [
    "../generated/cdrs/seq/human_cdr_seq.tsv",
    "../generated/cdrs/seq/influenza_cdr_seq.tsv",
    "../generated/cdrs/seq/corona_cdr_seq.tsv"
]

# Go through all files
for file in files: # Loop over all files in files
    df = pd.read_csv(file, sep="\t") # loads the file as a Pandas DataFrame

    # Check and clean up CDR columns
    for cdr in ['CDR_H1', 'CDR_H2', 'CDR_H3']: # defines loop over the three CDR regions
        if cdr in df.columns:
            # Keep only valid sequences
            df = df[df[cdr].apply(is_valid_sequence)]
            print(f"   {cdr}: {len(df)} valid sequences")

            # Count amino acids and add new columns
            for index, row in df.iterrows():
                seq = row[cdr]
                for aa in amino_acids:
                    count = seq.count(aa)
                    df.loc[index, f'{cdr}_count_{aa}'] = count # Saves the result in a new column
        else:
            print(f"   column {cdr} is missing in {file}")

    # Directly overwrite the original file
    df.to_csv(file, sep="\t", index=False)
    print(f"   File updated and saved: {file}") # Confirmation: File was successfully overwritten


   CDR_H1: 396 valid sequences
   CDR_H2: 396 valid sequences
   CDR_H3: 396 valid sequences
   File updated and saved: ../generated/cdrs/seq/human_cdr_seq.tsv
   CDR_H1: 98 valid sequences
   CDR_H2: 98 valid sequences
   CDR_H3: 98 valid sequences
   File updated and saved: ../generated/cdrs/seq/influenza_cdr_seq.tsv
   CDR_H1: 467 valid sequences
   CDR_H2: 467 valid sequences
   CDR_H3: 467 valid sequences
   File updated and saved: ../generated/cdrs/seq/corona_cdr_seq.tsv


Erklärung:

1. Vorbereitung

- amino_acids ist eine Liste aller Aminosäuren-Buchstaben (z. B. 'A', 'R', 'N', …).

- is_valid_sequence ist eine Funktion, die prüft, ob eine Sequenz gültig ist (kein "nan", "none" oder leer)

2. Dateien, die verarbeitet werden

dateien = [
    "human_cdr_seq.tsv",
    "influenza_cdr_seq.tsv",
    "corona_cdr_seq.tsv"
]
= TSV-Dateien werden nacheinander verarbeitet.

3. Für jede Datei wird gemacht:

a. Einlesen der Datei    
- df = pd.read_csv(datei, sep="\t")


b. Für jede CDR-Region (CDR_H1, CDR_H2, CDR_H3):
- Nur gültige Sequenzen behalten (is_valid_sequence)
- Aminosäuren zählen: Für jede gültige Sequenz wird gezählt, wie oft jede Aminosäure vorkommt.
→ Die Zählwerte werden in neuen Spalten gespeichert, z. B. CDR_H3_count_A.

c. Ergebnis speichern
- df.to_csv(f"bereinigt_{datei}", sep="\t", index=False)

 Die bereinigte und erweiterte Tabelle wird unter einem neuen Namen gespeichert

### Calculate and Clean Average Amino Acid Properties in CDR Sequences

In [3]:
for file in files:
    print(f"\nCalculate average properties for: {file}")
    df = pd.read_csv(file, sep="\t")

    for cdr in ['CDR_H1', 'CDR_H2', 'CDR_H3']: # Outer loop over the three CDR regions
        if cdr in df.columns:
            for index, row in df.iterrows(): # Iterates through each row of the table
                sequence = row[cdr] # Retrieves the amino acid sequence of the current line

                if isinstance(sequence, str): # Checks whether sequence is really a string (no NaN, float etc.)
                    hydrophobicities = []
                    charges = []
                    polarities = []
                    masses = []

                    for aa in sequence:
                        if aa in amino_acid_properties:
                            hydrophobicities.append(amino_acid_properties[aa]['hydrophobicity'])
                            charges.append(amino_acid_properties[aa]['charge'])
                            polarities.append(amino_acid_properties[aa]['polarity'])
                            masses.append(amino_acid_properties[aa]['mass'])
                        else:
                            print(f" Unknown amino acid '{aa}' in {cdr}, row {index+1}")

                    #  Calculate mean values
                    if hydrophobicities:
                        df.loc[index, f'{cdr}_mean_hydrophobicity'] = sum(hydrophobicities) / len(hydrophobicities)
                    if charges:
                        df.loc[index, f'{cdr}_mean_charge'] = sum(charges) / len(charges)
                    if polarities:
                        df.loc[index, f'{cdr}_mean_polarity'] = sum(polarities) / len(polarities)
                    if masses:
                        df.loc[index, f'{cdr}_mean_mass'] = sum(masses) / len(masses)
                else:
                    print(f" invalid {cdr}-Sequence in row {index+1} (Typ: {type(sequence)})")



    # Saves directly back to the original file
    df.to_csv(file, sep="\t", index=False)
    print(f"Properties directly in {file} saved (original file extended)")




Calculate average properties for: ../generated/cdrs/seq/human_cdr_seq.tsv
Properties directly in ../generated/cdrs/seq/human_cdr_seq.tsv saved (original file extended)

Calculate average properties for: ../generated/cdrs/seq/influenza_cdr_seq.tsv
Properties directly in ../generated/cdrs/seq/influenza_cdr_seq.tsv saved (original file extended)

Calculate average properties for: ../generated/cdrs/seq/corona_cdr_seq.tsv
Properties directly in ../generated/cdrs/seq/corona_cdr_seq.tsv saved (original file extended)


### Calculate and Add CDR Lengths to Existing Antibody Sequence Files

In [4]:
# Calculate CDR lengths and write directly to existing files
cdrs = ["CDR_H1", "CDR_H2", "CDR_H3"]

for file in files:
    print(f"\nCalculate CDR lengths for: {file}")
    df = pd.read_csv(file, sep="\t") #Loads the TSV file as Panda's DataFrame

    for cdr in cdrs:
        if cdr in df.columns:
            print(f" -> Processed: {cdr}") # Informs the terminal that this CDR is being processed
            df = df[df[cdr].apply(is_valid_sequence)]  # Only valid sequences
            df[f'{cdr.lower()}_length'] = df[cdr].apply(len) # New column is created
        else:
            print(f" {cdr} not available in {file}")

    df.to_csv(file, sep="\t", index=False)
    print(f"Saved updated: {file}")



Calculate CDR lengths for: ../generated/cdrs/seq/human_cdr_seq.tsv
 -> Processed: CDR_H1
 -> Processed: CDR_H2
 -> Processed: CDR_H3
Saved updated: ../generated/cdrs/seq/human_cdr_seq.tsv

Calculate CDR lengths for: ../generated/cdrs/seq/influenza_cdr_seq.tsv
 -> Processed: CDR_H1
 -> Processed: CDR_H2
 -> Processed: CDR_H3
Saved updated: ../generated/cdrs/seq/influenza_cdr_seq.tsv

Calculate CDR lengths for: ../generated/cdrs/seq/corona_cdr_seq.tsv
 -> Processed: CDR_H1
 -> Processed: CDR_H2
 -> Processed: CDR_H3
Saved updated: ../generated/cdrs/seq/corona_cdr_seq.tsv


### control

In [5]:
# Load data set (e.g. SARS-CoV-2)
df = pd.read_csv( "../generated/cdrs/seq/corona_cdr_seq.tsv", sep="\t")

# Show only the first 5 lines
df.head(5)

,pdb,heavy_chain,CDR_H3,CDR_H2,CDR_H1,CDR_H1_count_A,CDR_H1_count_R,CDR_H1_count_N,CDR_H1_count_D,CDR_H1_count_C,...,CDR_H2_mean_charge,CDR_H2_mean_polarity,CDR_H2_mean_mass,CDR_H3_mean_hydrophobicity,CDR_H3_mean_charge,CDR_H3_mean_polarity,CDR_H3_mean_mass,cdr_h1_length,cdr_h2_length,cdr_h3_length
0,9cci,B,TFGTYYDNTEDWFFDF,YGGDSD,GYSFSSF,0.0,0.0,0.0,0.0,0.0,...,-0.333333,9.900000,105.090000,-0.768750,-0.250000,8.518750,129.261250,7,6,16
1,9ccj,H,LPLGERIDY,YYSGT,GGSINTNMY,0.0,0.0,2.0,0.0,0.0,...,0.000000,7.840000,117.920000,-0.300000,-0.111111,8.222222,119.470000,9,5,9
2,9bj2,H,HNGDPYDFWSGYNTWAGGLDV,IPFLDV,GVIFSRN,0.0,1.0,1.0,0.0,0.0,...,-0.166667,7.033333,114.140000,-0.819048,-0.095238,8.652381,115.499524,7,6,21
3,9bj3,C,VPQAGAAQGHYYYYYGMDV,IPILGI,GGTFINY,0.0,0.0,1.0,0.0,0.0,...,0.000000,6.250000,104.135000,-0.384211,0.000000,8.010526,115.229474,7,6,19
4,8z6r,E,QGDLGDWILLGY,YPGDSD,GYTFSYY,0.0,0.0,0.0,0.0,0.0,...,-0.333333,9.733333,108.768333,0.166667,-0.166667,7.916667,115.458333,7,6,12


### see which columns we have now

In [6]:
# Show all column names
print(df.columns.tolist())


['pdb', 'heavy_chain', 'CDR_H3', 'CDR_H2', 'CDR_H1', 'CDR_H1_count_A', 'CDR_H1_count_R', 'CDR_H1_count_N', 'CDR_H1_count_D', 'CDR_H1_count_C', 'CDR_H1_count_Q', 'CDR_H1_count_E', 'CDR_H1_count_G', 'CDR_H1_count_H', 'CDR_H1_count_I', 'CDR_H1_count_L', 'CDR_H1_count_K', 'CDR_H1_count_M', 'CDR_H1_count_F', 'CDR_H1_count_P', 'CDR_H1_count_S', 'CDR_H1_count_T', 'CDR_H1_count_W', 'CDR_H1_count_Y', 'CDR_H1_count_V', 'CDR_H2_count_A', 'CDR_H2_count_R', 'CDR_H2_count_N', 'CDR_H2_count_D', 'CDR_H2_count_C', 'CDR_H2_count_Q', 'CDR_H2_count_E', 'CDR_H2_count_G', 'CDR_H2_count_H', 'CDR_H2_count_I', 'CDR_H2_count_L', 'CDR_H2_count_K', 'CDR_H2_count_M', 'CDR_H2_count_F', 'CDR_H2_count_P', 'CDR_H2_count_S', 'CDR_H2_count_T', 'CDR_H2_count_W', 'CDR_H2_count_Y', 'CDR_H2_count_V', 'CDR_H3_count_A', 'CDR_H3_count_R', 'CDR_H3_count_N', 'CDR_H3_count_D', 'CDR_H3_count_C', 'CDR_H3_count_Q', 'CDR_H3_count_E', 'CDR_H3_count_G', 'CDR_H3_count_H', 'CDR_H3_count_I', 'CDR_H3_count_L', 'CDR_H3_count_K', 'CDR_H3_cou